El script debe dar tratamiento conjuntos de datos grandes

  1.​ Tomar muestras de 5.000 y 10.000 registros.

  2.​ Calcular la matriz de distancias Gower para cada muestra.

  3.​ Medir tiempo de ejecución y uso de memoria.
  
  4.​ Comparar estabilidad de clusters en muestras de distinto tamaño.​

Muestra de 5000 registros:

In [ ]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
import time # Para medir el tiempo de ejecución
import psutil # Para medir el uso de memoria
import os # Para obtener el ID del proceso actual

# Configuración para mejor visualización en Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# =================================================================
# Carga del DataFrame desde archivos locales
# =================================================================
df = cargar_dataset_adult()

# =================================================================
# Implementación Vectorizada de Gower para una MUESTRA
# =================================================================

# 1. Preparación de los datos para Gower
# -----------------------------------------------------------------
# Define el tamaño de la muestra
sample_size = 5000

# Tomamos una muestra aleatoria del DataFrame
df_sample = df.sample(n=sample_size, random_state=42).copy()

print(f"--- Se ha tomado una muestra aleatoria de {sample_size} filas. ---")

# Identificar tipos de variables de forma definitiva
# Nota: fnlwgt (final weight) NO se incluye porque es un peso de muestreo del censo,
# no una característica descriptiva. No tiene sentido en el cálculo de similitud.
numeric_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# --- Manejo de la Variable Ordinal 'education' ---
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}


# 2. Cálculo Vectorizado de la Matriz de Distancias
# -----------------------------------------------------------------
print("\nCalculando la matriz de distancias (VERSIÓN VECTORIZADA)...")

# --- INICIO DE MEDICIÓN DE RECURSOS ---
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 * 1024) # Memoria en MB
start_time = time.time()
print(f"Uso de memoria antes del cálculo: {mem_before:.2f} MB")
# ------------------------------------


# Paso A: Separar los datos en arrays de NumPy por tipo
df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)

# Paso B: Calcular la disimilitud para cada tipo de variable
numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

# Paso C: Sumar todas las disimilitudes (numerador de Gower)
total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim

# Paso D: Calcular los pesos (denominador de Gower) de forma vectorizada
is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

# Paso E: Calcular la matriz de Gower final, evitando la división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix_vectorized = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)


# --- FIN DE MEDICIÓN DE RECURSOS ---
end_time = time.time()
mem_after = process.memory_info().rss / (1024 * 1024) # Memoria en MB

print("\n--- Resumen de Ejecución (Vectorizado) ---")
print(f"¡Cálculo completado en {end_time - start_time:.4f} segundos!")
print(f"Uso de memoria final: {mem_after:.2f} MB")
print(f"Memoria adicional utilizada por el cálculo: {mem_after - mem_before:.2f} MB")
# ---------------------------------


# 3. Visualización de Resultados
# -----------------------------------------------------------------
gower_matrix_df = pd.DataFrame(gower_matrix_vectorized, index=df_sample.index, columns=df_sample.index)
print("\n--- Matriz de Distancias de Gower resultante (versión vectorizada) ---")
print("(Mostrando las primeras 10x10 para visualización)")

# Retornar el DataFrame lo mostrará en formato tabla
gower_matrix_df.iloc[:10, :10]

Muestra de 10.000 registros:

In [ ]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
import time # Para medir el tiempo de ejecución
import psutil # Para medir el uso de memoria
import os # Para obtener el ID del proceso actual

# Configuración para mejor visualización en Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# =================================================================
# Carga del DataFrame desde archivos locales
# =================================================================
df = cargar_dataset_adult()

# =================================================================
# Implementación Vectorizada de Gower para una MUESTRA
# =================================================================

# 1. Preparación de los datos para Gower
# -----------------------------------------------------------------
# Define el tamaño de la muestra
sample_size = 10000

# Tomamos una muestra aleatoria del DataFrame
df_sample = df.sample(n=sample_size, random_state=42).copy()

print(f"--- Se ha tomado una muestra aleatoria de {sample_size} filas. ---")

# Identificar tipos de variables de forma definitiva
# Nota: fnlwgt (final weight) NO se incluye porque es un peso de muestreo del censo,
# no una característica descriptiva. No tiene sentido en el cálculo de similitud.
numeric_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# --- Manejo de la Variable Ordinal 'education' ---
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}


# 2. Cálculo Vectorizado de la Matriz de Distancias
# -----------------------------------------------------------------
print("\nCalculando la matriz de distancias (VERSIÓN VECTORIZADA)...")

# --- INICIO DE MEDICIÓN DE RECURSOS ---
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 * 1024) # Memoria en MB
start_time = time.time()
print(f"Uso de memoria antes del cálculo: {mem_before:.2f} MB")
# ------------------------------------


# Paso A: Separar los datos en arrays de NumPy por tipo
df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)

# Paso B: Calcular la disimilitud para cada tipo de variable
numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

# Paso C: Sumar todas las disimilitudes (numerador de Gower)
total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim

# Paso D: Calcular los pesos (denominador de Gower) de forma vectorizada
is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

# Paso E: Calcular la matriz de Gower final, evitando la división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix_vectorized = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)


# --- FIN DE MEDICIÓN DE RECURSOS ---
end_time = time.time()
mem_after = process.memory_info().rss / (1024 * 1024) # Memoria en MB

print("\n--- Resumen de Ejecución (Vectorizado) ---")
print(f"¡Cálculo completado en {end_time - start_time:.4f} segundos!")
print(f"Uso de memoria final: {mem_after:.2f} MB")
print(f"Memoria adicional utilizada por el cálculo: {mem_after - mem_before:.2f} MB")
# ---------------------------------


# 3. Visualización de Resultados
# -----------------------------------------------------------------
gower_matrix_df = pd.DataFrame(gower_matrix_vectorized, index=df_sample.index, columns=df_sample.index)
print("\n--- Matriz de Distancias de Gower resultante (versión vectorizada) ---")
print("(Mostrando las primeras 10x10 para visualización)")

# Retornar el DataFrame lo mostrará en formato tabla
gower_matrix_df.iloc[:10, :10]